# 01 · 산모를 기준으로 평가 대상을 나눈다

새 산모에 대한 성능을 보려면 같은 산모의 기록을 한 집합에 묶어야 한다. 반복 검사나 쌍태아 기록이 학습과 시험에 흩어지면 이미 본 산모의 특성이 평가에 섞일 수 있다. 이 노트북은 모델을 학습하기 전에 그 경계를 정한다.

$$D=\{(x_i,y_i,m_i,r_i,s_i)\}_{i=1}^{n},\qquad
x_i\in\mathbb R^{2\times150},\quad y_i\in\{0,1\}.$$

구간 $i$에서 $m_i$는 산모, $r_i$는 검사 기록, $s_i$는 사이트 코드다. $x_i$에는 FHR과 TOCO가 채널별로 150개씩 들어 있다. 측정 간격은 2초이며 구간 길이는 5분이다. $y_i=1$은 전문가의 비정상 판독을 뜻한다.

이 노트북이 있는 폴더에서 실행한다. Python 3.10 이상과 NumPy, pandas, scikit-learn, Jupyter가 필요하다. 입력은 00이 만든 `data/prepared/`에서 읽고, 분할 결과는 `data/results/`에 저장한다. 입력이 없으면 00부터 실행한다.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

# Read the prepared data from 00 and save the split for later experiments.
DATASET = Path("data/prepared")
RESULTS = Path("data/results")
SEED = 42

### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

## 1. 표의 행과 신호의 행을 맞춘다

00에서 확인한 원본 ID·라벨 대응, 신호 구조, 결측 기준을 바탕으로 분할한다. 데이터 규모와 제외 사유는 00에서 확인하고, 여기서는 구간 표와 신호 배열의 연결을 검사한다.

segments.csv의 signal_index는 raw_signals.npy의 행을 가리킨다. ID는 문자열로 읽어 앞자리 0을 지키고, 신호는 읽기 전용으로 연다. 구간 ID 중복·배열 크기·인덱스 순서가 어긋나면 분석을 멈춘다. 이 검사는 구조를 확인하며, 원본과 신호 내용의 대응을 다시 검증하지는 않는다.

In [ ]:
metadata = json.loads((DATASET / "dataset.json").read_text(encoding="utf-8"))

# Read IDs as strings: "00123" must not become 123.
id_types = {key: str for key in ("row_id", "record_id", "mother_id", "site")}
window_table = pd.read_csv(
    DATASET / "segments.csv",
    dtype=id_types,
    keep_default_na=False,
)
signal_array = np.load(DATASET / "raw_signals.npy", mmap_mode="r")

# Each table row must point to one (2, 150) FHR/TOCO array.
expected_shape = (len(window_table), 2, 150)
assert window_table["row_id"].is_unique, "구간 ID가 중복됩니다"
assert signal_array.shape == expected_shape, "신호 배열 크기가 구간 표와 다릅니다"

# Example: three retained segments have signal_index=[0, 1, 2].
expected_indices = np.arange(len(window_table))
assert np.array_equal(
    window_table["signal_index"], expected_indices
), "신호 인덱스 순서가 배열 행 순서와 다릅니다"

In [ ]:
# Report the array layout without selecting a participant's waveform.
print("segment count =", len(window_table))
print("signal shape =", signal_array.shape)

## 2. 산모 목록을 분할의 기준으로 삼는다

분할에 필요한 산모 목록은 이미 전처리 결과에 있다. `mothers.csv`는 원본 EMR의 `Mother.de-identification_ID`를 기준으로 남은 구간을 묶은 표다. 각 행에는 산모 ID, 기록 수, 구간 수, 비정상 구간 유무가 들어 있다.

이 산모 집합을 $M$이라 하자. 반복 검사나 쌍태아 기록이 있어도 산모는 한 번만 등장한다. 아래에서는 ID 중복이 없는지, 구간 표와 산모 집합이 같은지, 산모별 구간 수의 합이 전체 구간 수와 맞는지 확인한다.

In [ ]:
mother_table = pd.read_csv(
    DATASET / "mothers.csv",
    dtype={"mother_id": str},
    keep_default_na=False,
)

# Fictional twins: "DEMO527183_1" and "DEMO527183_2" share mother "DEMO527183".
listed_mother_ids = set(mother_table["mother_id"])
segment_mother_ids = set(window_table["mother_id"])
assert mother_table["mother_id"].is_unique, "산모 ID가 중복됩니다"
assert listed_mother_ids == segment_mother_ids, "산모 목록이 다릅니다"
assert mother_table["n_windows"].sum() == len(window_table), "산모별 구간 수의 합이 다릅니다"

In [ ]:
# Counts include all mothers and retained windows, without listing individual IDs.
print("mother count =", len(mother_table))
print("segment count =", mother_table["n_windows"].sum())

## 3. 비율은 맞추되 산모는 쪼개지 않는다

학습·검증·시험의 목표 비율은 산모 수 기준 80:10:10이다. 먼저 전체의 10%를 시험에 배정하고, 남은 산모의 $1/9$을 검증에 배정한다. 인원은 정수이므로 실제 비율에는 반올림 차이가 생긴다.

비정상 구간을 하나라도 가진 산모인지에 따라 층화한다. 기준은 $b_m=\max_{i:m_i=m}y_i$이며, 산모 표의 `any_abnormal`에 해당한다. 이 방식은 산모의 비정상 여부를 고려하지만 각 집합의 비정상 구간 비율까지 같게 만들지는 않는다.

$$M=M_{\rm tr}\cup M_{\rm va}\cup M_{\rm te},\qquad
M_a\cap M_b=\varnothing\quad(a\ne b).$$

아래 첨자는 학습·검증·시험 집합을 뜻한다. 각 집합의 산모는 겹칠 수 없다. ID 순서를 정렬하고 난수 시드를 고정해 배정을 재현한다. 분할에 실패해도 시드를 바꿔 재시도하지 않는다.

In [ ]:
def mother_split(mothers, fraction, seed):
    # Sort IDs so the same seed sees the same input order.
    mothers = mothers.sort_values("mother_id")

    # Example: fraction=0.1 holds out 10 of 100 mothers.
    # Stratify by whether a mother has any abnormal retained segment.
    remaining_mothers, held_out_mothers = train_test_split(
        mothers,
        test_size=fraction,
        stratify=mothers["any_abnormal"],
        random_state=seed,
    )
    return remaining_mothers, held_out_mothers

분할은 세 집합이 모두 조건을 충족해야 성공이다. 각 집합에 정상·비정상 구간이 모두 있고, 모든 구간이 배정됐으며, 한 산모가 한 집합에만 속하는지 확인한다. 실패하면 부분 배정을 모두 취소해 불완전한 결과가 남지 않게 한다.

In [ ]:
window_table["split"] = "unavailable"
split_ready = False
split_error = None

try:
    if mother_table.empty:
        raise ValueError("남은 산모가 없습니다")

    # First hold out 10% for test, then 1/9 of the remaining 90% for validation.
    # Example: 100 mothers -> about 80 train, 10 val, and 10 test.
    development_mothers, test_mothers = mother_split(mother_table, 0.1, SEED)
    train_mothers, val_mothers = mother_split(development_mothers, 1 / 9, SEED + 1)
    mother_groups = {"train": train_mothers, "val": val_mothers, "test": test_mothers}

    # Apply each mother's assignment to all of her segments.
    for split_name, mothers in mother_groups.items():
        selected_windows = window_table["mother_id"].isin(mothers["mother_id"])
        n_classes = window_table.loc[selected_windows, "target"].nunique()
        if n_classes != 2:
            raise ValueError(f"{split_name} 집합에는 정상·비정상 구간이 모두 필요합니다")
        window_table.loc[selected_windows, "split"] = split_name

    # Every segment is assigned, and each mother appears in just one split.
    assert window_table["split"].ne("unavailable").all(), "배정되지 않은 구간이 있습니다"
    splits_per_mother = window_table.groupby("mother_id")["split"].nunique()
    assert splits_per_mother.eq(1).all(), "같은 산모가 여러 집합에 속합니다"
    split_ready = True

except (ValueError, AssertionError) as error:
    # A failed check cancels every assignment, including those already made.
    window_table["split"] = "unavailable"
    split_error = str(error)

In [ ]:
# Example: every segment for mother "DEMO846529" receives the same split.
# A failed split leaves every assignment as "unavailable".
mother_splits = window_table[["mother_id", "split"]].drop_duplicates()
print("split_ready =", split_ready)

### 3.1. 집합별 규모를 확인한다

학습 집합에서는 모델과 결측 대체·척도 변환의 기준을 추정한다. 검증 집합에서는 모델과 경보 임계값을 고른다. 시험 집합에서는 선택을 고정한 뒤 평가한다. 시험 결과를 보고 선택을 바꾸면 그 집합도 개발에 사용한 셈이다.

아래 표의 비정상 산모 수는 남은 비정상 구간이 하나라도 있는 산모를 센다. 비정상 구간 비율의 분모는 각 집합의 전체 구간 수다. 산모 단위로 층화해도 구간 비율은 집합마다 다를 수 있다.


In [ ]:
summary = window_table.groupby("split").agg(
    산모_수=("mother_id", "nunique"),
    기록_수=("record_id", "nunique"),
    구간_수=("target", "size"),
    비정상_구간_수=("target", "sum"),
)

# Count a mother once even if several of her segments are abnormal.
abnormal_windows = window_table.loc[window_table["target"].eq(1)]
abnormal_mothers = abnormal_windows.groupby("split")["mother_id"].nunique()
summary["비정상_산모_수"] = abnormal_mothers.reindex(summary.index, fill_value=0)

# Example: 20 abnormal segments among 100 segments -> 20%.
summary["비정상_구간_비율"] = summary["비정상_구간_수"] / summary["구간_수"]
display(summary.rename_axis("집합").style.format({"비정상_구간_비율": "{:.1%}"}))

### 3.2. 사이트별 규모를 확인한다

사이트 코드는 원본 `code`의 첫 접두사다. 아래 표는 모든 분할을 합친 사이트별 규모를 보여준다. 실제 병원과의 대응은 확인되지 않았으므로 사이트별 집계를 곧바로 병원별 집계로 해석하지 않는다.


In [ ]:
# Count unique mothers and retained segments across all splits at each site.
site_summary = window_table.groupby("site").agg(
    산모_수=("mother_id", "nunique"),
    구간_수=("row_id", "size"),
)
display(site_summary.rename_axis("사이트 코드"))

## 4. 분할 결과도 data 안에 저장한다

`data/prepared/`에는 전처리한 신호와 표가 있고, `data/results/`에는 이번 단계에서 정한 분할이 있다. 후속 분석은 분할 표와 기존 신호 배열을 함께 읽는다.

`segments.csv`에는 구간별 `split` 열을 더하고, `mother_splits.csv`에는 산모별 배정을 저장한다. `protocol.json`은 시드, 결측 기준, 데이터 규모와 분할 성공 여부를 기록한다.

재실행하면 같은 결과 파일을 갱신한다. 00을 다시 실행했다면 01과 후속 분석도 다시 실행해야 한다. 실패한 분할은 `split_ready=False`, 배정값은 `unavailable`로 저장한다. 이 상태로는 성능을 평가하지 않는다.

In [ ]:
# Record both the split settings and whether the split succeeded.
# If it failed, split_error explains why the assignments are unavailable.
protocol = dict(
    metadata,
    split_seed=SEED,
    split_ready=split_ready,
    split_error=split_error,
    split_unit="mother_id",
    split_source="mothers.csv",
    n_segments=len(window_table),
    n_mothers=len(mother_table),
)

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)

# segments.csv keeps every segment row and adds its mother's split.
# mother_splits.csv stores one assignment per mother.
window_table.to_csv(RESULTS / "segments.csv", index=False)
mother_splits.to_csv(RESULTS / "mother_splits.csv", index=False)

protocol_path = RESULTS / "protocol.json"
protocol_path.write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8",
)
print("저장 위치:", RESULTS)

## 분할이 보장하는 범위를 밝힌다

분할에 성공하면 같은 산모의 반복 검사와 쌍태아 기록이 집합을 넘나들지 않는다. 다만 해당 산모가 과거 연구에 포함됐는지는 알 수 없다. 원본에 그 이력이 없기 때문이다.

집합별 규모는 위 표에서 확인한다. 이후 성능 비교에서는 저장한 배정을 고정하며, 분할에 실패하면 아래 사유를 확인한 뒤 입력 데이터를 점검한다.

In [ ]:
if split_ready:
    print("분할 완료. 후속 분석에서는 저장한 산모 배정을 사용합니다.")
else:
    print('분할 불가:', '[details omitted from notebook output]')

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts([] if split_error is None else [split_error])